# CLOAF on a GPU notebook (Colab or Kaggle)

**Colab:** Runtime → Change runtime type → GPU. **Kaggle:** Settings → Accelerator → GPU, and add the data as a Kaggle dataset instead of Drive (change `DATA` below).

Put the licensed data in Google Drive (never in the repo):

```
MyDrive/cloaf_data/
  SMPL_python_v.1.1.0.zip (from smpl.is.tue.mpg.de, as downloaded)
  amass/ACCAD.tar.bz2     (AMASS, "SMPL+H G") -> training
  amass/SFU.tar.bz2       (AMASS, "SMPL+H G") -> test sequences
```

Everything that matters (checkpoints, logs, results) is written to Drive, so a disconnect loses nothing:
reconnect, run the setup cells again and continue.

## Setup (run again after every disconnect)

In [ ]:
!nvidia-smi

In [ ]:
import os
os.chdir('/content')
if not os.path.exists('/content/cloaf_impl'):
    !git clone https://github.com/arhannlp/cloaf_impl.git /content/cloaf_impl
%cd /content/cloaf_impl
!git pull -q
!pip install -q torchdiffeq smplx trimesh

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, os, subprocess
DATA = '/content/drive/MyDrive/cloaf_data'
RES = f'{DATA}/results'
for d in ['data/body_models', 'data/amass', f'{DATA}/checkpoints', RES]:
    os.makedirs(d, exist_ok=True)
print(os.listdir(DATA), os.listdir(f'{DATA}/amass'))

# SMPL: official zip -> chumpy-free SMPL_*.pkl
!python scripts/convert_smpl_pkl.py --src {DATA}/SMPL_python_v.1.1.0.zip --out data/body_models/smpl

# AMASS: extract each archive once (a re-run only extracts archives added since, e.g. CMU)
for f in sorted(glob.glob(f'{DATA}/amass/*.tar.bz2')):
    done = f'data/amass/.{os.path.basename(f)}.done'
    if not os.path.exists(done):
        print('extracting', os.path.basename(f), '...')
        subprocess.run(['tar', '-xjf', f, '-C', 'data/amass'], check=True)
        open(done, 'w').close()
print('AMASS folders:', [d for d in os.listdir('data/amass') if not d.startswith('.')])
print('AMASS files:', len(glob.glob('data/amass/**/*.npz', recursive=True)))
assert os.path.exists('data/body_models/smpl/SMPL_NEUTRAL.pkl'), 'SMPL conversion failed: read the messages above'
print('setup OK')

## Step 0 – smoke test (fake body, must print SMOKE TEST OK) and real SMPL check (PASS lines)

In [ ]:
!python tests/smoke_test.py 2>&1 | grep -E 'PASS|FAIL|SKIP|SMOKE'
!python tests/check_smpl.py --smpl data/body_models/smpl 2>&1 | grep --line-buffered -v WARNING

## Step 1 – Jacobian check on the real SMPL (compare with Fig. 5 / B.7; write the numbers down)

In [ ]:
!python scripts/check_linearity.py --smpl data/body_models/smpl --restarts 3 --out {DATA}/results/linearity.npz

## Step 2 – simple target field, no training (hand-target distance must shrink)

In [ ]:
!python scripts/demo_target_field.py --smpl data/body_models/smpl --out {DATA}/results/target_field.npz

## Step 3 – prepare AMASS: train on ACCAD, test on SFU (held out)

In [ ]:
# Folders created by extracting the AMASS archives. To add CMU later (second training
# block), set TRAIN_SETS = ['ACCAD', 'CMU'], re-run the setup cell and this cell, then continue training.
TRAIN_SETS, TEST_SET = ['ACCAD'], 'SFU'
train_roots = [f'data/amass/{t}' for t in TRAIN_SETS if os.path.isdir(f'data/amass/{t}')] or ['data/amass']
test_root = f'data/amass/{TEST_SET}' if os.path.isdir(f'data/amass/{TEST_SET}') else train_roots[0]
ROOTS = ' '.join(train_roots)
!python scripts/prepare_amass.py --amass_root {ROOTS} --out data/amass_smpl_30fps.npz
SEQS = sorted(s for s in glob.glob(f'{test_root}/**/*.npz', recursive=True) if not s.endswith('shape.npz'))
TEST = [SEQS[i] for i in (0, len(SEQS) // 3, 2 * len(SEQS) // 3)]   # 3 different sequences
print('train on', train_roots, '| test on', test_root)
print(*TEST, sep='\n')

## Step 3b – collision test on clean AMASS (1–2 min)

Clean mocap should rarely collide. Compare the four configurations; the default is the last one.

In [ ]:
!python -u tests/check_collisions_amass.py --smpl data/body_models/smpl --amass data/amass_smpl_30fps.npz --frames 200 2>&1 | grep --line-buffered -v WARNING

## Step 4 – training-free baselines (run while you wait, or before training)

* `point_target`: Supp. C "simple field"; expected to keep the collisions (no non-intersection property).
* `smooth_target`: smooth in space, so touching points move together; our own training-free variant.

About 3–4 minutes per run on a T4 (6 runs). Progress is printed every 10 frames.

In [ ]:
for seq in TEST:
    name = os.path.basename(seq)[:-4]
    for fld in ['point_target', 'smooth_target']:
        out = f'{RES}/{name}_{fld}_all.npz'
        !python -u scripts/run_on_sequence.py --smpl data/body_models/smpl --seq "{seq}" --field {fld} --frames 90 --corrupt --mode all --out "{out}" 2>&1 | grep --line-buffered -v WARNING

## Step 5 – train the motion field

Re-run this cell to **continue** training (it resumes from Drive). Each run lasts `--minutes`.
The loss must go down; if it does not after ~20 minutes, stop and check.

In [ ]:
!python -u -m cloaf.train --smpl data/body_models/smpl --amass data/amass_smpl_30fps.npz \
    --batch 16 --minutes 120 --resume --out {DATA}/checkpoints/field.pt

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f'{DATA}/checkpoints/field.pt.log.csv')
plt.semilogy(log.iteration, log.loss); plt.xlabel('iteration'); plt.ylabel('loss (avg of 100)'); plt.grid(); plt.show()
log.tail()

In [ ]:
# Is the loss good? Compare the checkpoint with 'do nothing' (zero field) and the
# point-target field. A useful network is clearly below 1.00x, ideally close to the point target.
!python -u scripts/loss_reference.py --smpl data/body_models/smpl --amass data/amass_smpl_30fps.npz --field {DATA}/checkpoints/field.pt 2>&1 | grep --line-buffered -v WARNING

## Step 6 – CLOAF with the trained network, both modes (the project extension)

In [ ]:
CKPT = f'{DATA}/checkpoints/field.pt'
for seq in TEST:
    name = os.path.basename(seq)[:-4]
    for mode in ['all', 'colliding_only']:
        out = f'{RES}/{name}_net_{mode}.npz'
        !python -u scripts/run_on_sequence.py --smpl data/body_models/smpl --seq "{seq}" --field "{CKPT}" --frames 90 --corrupt --mode {mode} --out "{out}" 2>&1 | grep --line-buffered -v WARNING

## Step 7 – summary table (one row per run; also saved as results/summary_runs.csv in Drive)

In [ ]:
import pandas as pd
pd.read_csv(f'{RES}/summary_runs.csv')

## Correct any motion with the working pipeline

Put your file (AMASS `.npz`, or `.npz` with `body_pose`) in Drive, set `INPUT`, run. The GIF shows the input (left) and the CLOAF output (right), colliding faces in red.

In [ ]:
INPUT = 'data/amass/SFU/0005/0005_2FeetJump001_poses.npz'     # or f'{DATA}/my_motion.npz'
OUTPUT = f'{RES}/' + os.path.basename(INPUT).replace('.npz', '_cloaf.npz')
!python -u scripts/cloaf_correct.py --smpl data/body_models/smpl --input "{INPUT}" --output "{OUTPUT}" --frames 90 2>&1 | grep --line-buffered -v WARNING
!python -u scripts/render_gif.py --smpl data/body_models/smpl --result "{OUTPUT}" 2>&1 | grep -v WARNING
from IPython.display import Image
Image(OUTPUT.replace('.npz', '.gif'))

# Paper reproduction on 3DPW (Tables 1, 2, 4)

Run the setup cells at the top first (clone, Drive). Needs a GPU.
Drive: `cloaf_data/3DPW/sequenceFiles.zip` and `cloaf_data/3DPW/imageFiles.zip`.

## R1 – install HMR2.0 (4D-Humans) and extract 3DPW (once per Colab machine)

In [ ]:
import os, subprocess
if not os.path.exists('/content/4D-Humans'):
    !git clone -q https://github.com/shubham-goel/4D-Humans.git /content/4D-Humans
!pip install -q pytorch-lightning yacs timm einops scikit-image webdataset dill gdown pyrender
!pip install -q --no-deps -e /content/4D-Humans
# HMR2.0 uses the same SMPL neutral model (our chumpy-free conversion)
SMPL_CACHE = os.path.expanduser('~/.cache/4DHumans/data/smpl')
os.makedirs(SMPL_CACHE, exist_ok=True)
!cp -n data/body_models/smpl/SMPL_NEUTRAL.pkl {SMPL_CACHE}/SMPL_NEUTRAL.pkl
# 3DPW
os.makedirs('data/3DPW', exist_ok=True)
for z in ['sequenceFiles', 'imageFiles']:
    if not os.path.exists(f'data/3DPW/{z}'):
        print('extracting', z, '...', flush=True)
        subprocess.run(['unzip', '-q', '-n', f'{DATA}/3DPW/{z}.zip', '-d', 'data/3DPW'], check=True)
print(os.listdir('data/3DPW'), len(os.listdir('data/3DPW/sequenceFiles/test')), 'test sequences')

## R2 – quick test: HMR2.0 on one sequence, 32 frames per person (~2 min)

The first run downloads the HMR2.0 checkpoint (a few GB) into `~/.cache/4DHumans`.

In [ ]:
SEQ = sorted(os.listdir('data/3DPW/sequenceFiles/test'))[0][:-4]
!PYTHONPATH=/content/4D-Humans python -u scripts/hmr2_3dpw.py --pw3d data/3DPW --seqs {SEQ} --max_frames 32 --out results/hmr2_quick.npz 2>&1 | grep -v -i warning
!python -u scripts/eval_3dpw.py --smpl data/body_models/smpl --pw3d data/3DPW --pred results/hmr2_quick.npz --seqs {SEQ} 2>&1 | grep -v WARNING

### R2b. Which body parts collide? (calibration of Col.Rate)
Per frame: colliding vertices with the predicted shape and with betas = 0, the part pairs that collide, and a picture of the worst frame (red = colliding vertices).

In [ ]:
!python -u scripts/diagnose_collisions.py --smpl data/body_models/smpl --pred results/hmr2_quick.npz --key {SEQ}/0 --png results/diag.png 2>&1 | grep -v WARNING
from IPython.display import Image, display
display(Image('results/diag.png'))

## R3 – HMR2.0 on the whole 3DPW test set (Table 1 / 2, HMR2.0 row)

Resumable: the file in Drive is updated after every sequence; after a disconnect, run R1 and this cell again.

In [ ]:
PRED = f'{DATA}/results/hmr2_3dpw_test.npz'
!PYTHONPATH=/content/4D-Humans python -u scripts/hmr2_3dpw.py --pw3d data/3DPW --out "{PRED}" 2>&1 | grep -v -i warning
!python -u scripts/eval_3dpw.py --smpl data/body_models/smpl --pw3d data/3DPW --pred "{PRED}" --out "{DATA}/results/eval_hmr2.npz" 2>&1 | grep -v WARNING

### R3b. Evaluation only (if R3 stopped during the evaluation)
The predictions are already in Drive, so HMR2.0 is not run again. Resumable: finished tracks are kept in `eval_hmr2.partial.npz`. After a runtime reset, run the setup cells and R1 first (3DPW files). The joint regressor is copied to Drive once, so later runs need no download.

In [ ]:
import os
PRED = f'{DATA}/results/hmr2_3dpw_test.npz'
J19 = f'{DATA}/SMPL_to_J19.pkl'
assert os.path.isdir('data/3DPW/sequenceFiles'), 'run R1 first (it extracts 3DPW)'
if not os.path.exists(J19):
    src = os.path.expanduser('~/.cache/4DHumans/data/SMPL_to_J19.pkl')
    if not os.path.exists(src):   # extract only this file from the HMR2.0 data archive (~2.5 GB download, ~5 min)
        os.makedirs(os.path.expanduser('~/.cache/4DHumans'), exist_ok=True)
        !curl -L -o /content/hmr2_data.tar.gz https://www.cs.utexas.edu/~pavlakos/4dhumans/hmr2_data.tar.gz
        !tar -xf /content/hmr2_data.tar.gz -C ~/.cache/4DHumans data/SMPL_to_J19.pkl && rm /content/hmr2_data.tar.gz
    !cp "{src}" "{J19}"
print('joint regressor:', os.path.exists(J19))
!python -u scripts/eval_3dpw.py --smpl data/body_models/smpl --pw3d data/3DPW --pred "{PRED}" --j19 "{J19}" --out "{DATA}/results/eval_hmr2.npz" 2>&1 | grep -v WARNING

## Google Cloud training (GPU VM)
### G1. Copy the data from Drive to a Cloud Storage bucket (once)
Create the bucket first (Console → Cloud Storage → Buckets → Create, region europe-west4, public access prevention on). Fill in your project ID and bucket name. Then on the VM: `bash scripts/gcp_train.sh gs://BUCKET 600`.

In [ ]:
PROJECT = 'my-project-id'          # Console, top bar: project selector -> ID column
BUCKET = 'gs://cloaf-data-CHANGE-ME'
from google.colab import auth
auth.authenticate_user()
!gcloud config set project {PROJECT}
!gsutil -m cp "{DATA}/SMPL_python_v.1.1.0.zip" {BUCKET}/
!gsutil -m cp "{DATA}/amass/*.tar.bz2" {BUCKET}/amass/
!gsutil -m cp "{DATA}/checkpoints/field.pt*" {BUCKET}/checkpoints/ || echo 'no checkpoint yet'
!gsutil ls -l -r {BUCKET}

### G2. Copy the trained checkpoint back to Drive (after training)

In [ ]:
!gsutil -m cp "{BUCKET}/checkpoints/*" "{DATA}/checkpoints/"
!cat "{DATA}/checkpoints/loss_reference.txt"